# Slab2: Notebook overview

This notebook uses the separate `py-gdmate-pygmt` environment described in the **Optional PyGMT environment** section of the README, rather than the default `py-gdmate` environment.

This notebook demonstrates how to download and cache the Slab2 dataset.

It also covers how to use the same approach to work with other global datasets (e.g., seafloor age, relief, and plate boundaries) and combine them into map plots.

The two examples demonstrate regional and global plots, respectively. The plots involve many specific configuration options. The purpose of this notebook is not to explain all these options in detail, but to demonstrate how to combine multiple datasets to generate near-publication-quality maps.

The plotting details are encapsulated in functions such as `plot_slab2_global`. Users can explore the implementations of these functions in separate files within the GDMATE repository.

# Slab2: Regional grids

Download published Slab2 grids, then plot regional maps of depth, dip and strike.

Source: [USGS Slab2 grids](https://apps.usgs.gov/shakemap_geodata/slabs/).
Citation: Hayes (2018), [Slab2 data release](https://doi.org/10.5066/F7PV6JNV).
The downloader discovers dated filenames from the USGS listing and saves them.
The first example uses Tonga–Kermadec (`ker`) depth, dip and strike grids.

Existing cached files are reused offline unless `overwrite_downloads = True`.

In [ ]:
from pathlib import Path

from gdmate.slab2 import download_slab2

region = "ker"  # USGS region code; e.g. "cas" for Cascadia.
fields = ("depth", "dip", "strike")  # Also available: "uncertainty".
overwrite_downloads = False
network_timeout_s = 30

# Keep the default cache in the repository's ignored notebook data directory.
# Replace data_dir with any preferred local path.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "gdmate").is_dir()
)
data_dir = repo_root / "notebooks/data/slab2"
print(f"Region: {region}; fields: {fields}")
print(f"Download directory: {data_dir}")

## Download and inspect file records

A single download produces a record including the source URL, local path, size count and whether it came from cache instead. 

If a request fails, the error includes the USGS URL. Check network access and rerun.

Earlier successful files remain cached if a later download fails. 
Cached files do not check for upstream updates; use `overwrite_downloads = True` to request a refresh.

In [ ]:
downloaded_grids = download_slab2(
    region, data_dir, fields=fields, overwrite=overwrite_downloads,
    timeout=network_timeout_s,
)
for record in downloaded_grids:
    status = "cached" if record["cached"] else "downloaded"
    print(f"{record['field']}: {status}, {record['size_bytes']:,} bytes")
    print(f"  URL:  {record['url']}")
    print(f"  File: {record['path']}")

## Save metadata

A JSON metadata records the source URLs and location of local files.
The next step will reuse these metadata.

In [ ]:
import json

if downloaded_grids:
    manifest = {
        "citation": "Hayes (2018), https://doi.org/10.5066/F7PV6JNV",
        "region": region,
        "files": [
            dict(record, path=str(record["path"])) for record in downloaded_grids
        ],
    }
    manifest_file = data_dir / f"{region}_download_manifest.json"
    manifest_file.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"Saved manifest: {manifest_file}")

## Plot regional depth

The map shows positive-down depth in kilometres. Set
`download_seafloor_age_enabled = True` to fetch the regional age snapshot
from the [GMT EarthByte dataset](https://www.generic-mapping-tools.org/remote-datasets/earth-age.html)
([Seton et al., 2020](https://doi.org/10.1029/2020GC009214)). The maps use the
snapshot when it is cached.

In [ ]:
download_seafloor_age_enabled = False
figure_dir = data_dir / "figures"

In [ ]:
from gdmate.slab2 import load_seafloor_age, load_slab2_depth, plot_slab2_depth

regional_grids = {record["field"]: record["path"] for record in downloaded_grids}
depth_grid_file = regional_grids["depth"]
depth = load_slab2_depth(depth_grid_file)
y, x = depth.dims
regional_bounds = [
    float(depth[x].min()), float(depth[x].max()),
    float(depth[y].min()), float(depth[y].max()),
]
try:
    seafloor_age = load_seafloor_age(
        regional_bounds, data_dir, download=download_seafloor_age_enabled,
    )
except FileNotFoundError:
    if download_seafloor_age_enabled:
        raise
    seafloor_age = None

print(f"Depth grid: {depth_grid_file}")
slab_figure = plot_slab2_depth(
    depth_grid_file, region=regional_bounds, seafloor_age=seafloor_age,
)
figure_dir.mkdir(parents=True, exist_ok=True)
for suffix in ("png", "pdf"):
    figure_file = figure_dir / f"{region}_slab2_depth.{suffix}"
    slab_figure.savefig(figure_file)
    print(f"Saved: {figure_file}")
slab_figure.show()

## Plot regional dip and strike

Dip is shown from 0 to 90 degrees with contours. Strike uses a cyclic
color scale from 0 to 360 degrees.

In [ ]:
from gdmate.slab2 import plot_slab2_angle

for field in ("dip", "strike"):
    angle_figure = plot_slab2_angle(
        regional_grids[field], field=field, region=regional_bounds,
        seafloor_age=seafloor_age,
    )
    for suffix in ("png", "pdf"):
        figure_file = figure_dir / f"{region}_slab2_{field}.{suffix}"
        angle_figure.savefig(figure_file)
        print(f"Saved: {figure_file}")
    angle_figure.show()

## Download all Slab2 regions

This separate workflow discovers every region with a depth grid in the official
USGS directory and calls the same `download_slab2` downloader for each region.
Depth, dip and strike are requested by default; add `uncertainty` to
`global_fields` if needed. This covers grids available from that directory,
not other file types in the complete Slab2 release.

Set `global_download_enabled = True` and run the next cell. The download saves 
the data files from the server. Existing cached files are reused. 
Each region reports completion or failure, and a metadata record is
saved even if a region fails. Rerun to retry failures; the cell raises a summary
error after attempting every region. 

The global plotting block could reads its metadata records without requiring another network request.


In [ ]:
global_download_enabled = False
global_fields = ("depth", "dip", "strike")
global_overwrite_downloads = False
global_manifest_file = data_dir / "global_download_manifest.json"

if global_download_enabled:
    import json

    from gdmate.slab2 import list_slab2_regions

    global_regions = list_slab2_regions(timeout=network_timeout_s)
    global_records = []
    global_failures = {}
    for index, code in enumerate(global_regions, start=1):
        try:
            records = download_slab2(
                code, data_dir, fields=global_fields,
                overwrite=global_overwrite_downloads, timeout=network_timeout_s,
            )
            global_records.extend(dict(record, region=code) for record in records)
            print(f"[{index}/{len(global_regions)}] {code}: complete "
                  f"({sum(record['cached'] for record in records)} cached)")
        except (OSError, RuntimeError, ValueError) as error:
            global_failures[code] = str(error)
            print(f"[{index}/{len(global_regions)}] {code}: FAILED: {error}")
    global_manifest = {
        "citation": "Hayes (2018), https://doi.org/10.5066/F7PV6JNV",
        "regions": global_regions,
        "fields": list(global_fields),
        "files": [dict(record, path=str(record["path"])) for record in global_records],
        "failures": global_failures,
    }
    data_dir.mkdir(parents=True, exist_ok=True)
    global_manifest_file.write_text(
        json.dumps(global_manifest, indent=2) + "\n", encoding="utf-8",
    )
    print(f"Saved: {global_manifest_file}")
    if global_failures:
        raise RuntimeError(f"Downloads failed for: {', '.join(global_failures)}")
    print(f"Completed {len(global_regions)} regions, {len(global_records)} files.")
else:
    print("All-region download disabled. Set global_download_enabled = True.")

## Global slab overview

The global maps use the all-region manifest and three cached overlays.
Enable each download for the first run and turn it off to reuse its snapshot.

- `global_download_age_enabled` fetches the
  [GMT EarthByte seafloor-age grid](https://www.generic-mapping-tools.org/remote-datasets/earth-age.html).
- `global_download_relief_enabled` fetches
  [GMT Earth Relief](https://www.generic-mapping-tools.org/remote-datasets/earth-relief.html).
- `global_download_boundaries_enabled` fetches plate boundaries from
  [fraxen/tectonicplates](https://github.com/fraxen/tectonicplates), based on
  [Bird (2003), PB2002](https://doi.org/10.1029/2001GC000252). The dataset is
  distributed under the [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1.0/).

In [ ]:
global_download_age_enabled = False
global_download_relief_enabled = False
global_download_boundaries_enabled = False

if global_manifest_file.is_file():
    import json

    from gdmate.plate_boundaries import load_plate_boundaries
    from gdmate.slab2 import load_earth_relief, load_seafloor_age, plot_slab2_global

    global_manifest = json.loads(global_manifest_file.read_text(encoding="utf-8"))
    if global_manifest["failures"]:
        raise ValueError("Complete the failed downloads before plotting.")
    global_grids = {}
    for field in ("depth", "dip", "strike"):
        records = [record for record in global_manifest["files"]
                   if record["field"] == field]
        if {record["region"] for record in records} != set(global_manifest["regions"]):
            raise ValueError(
                f"The manifest lacks {field} grids. Rerun the all-region download."
            )
        global_grids[field] = [record["path"] for record in records]

    global_bounds = [0, 360, -65, 65]
    global_seafloor_age = load_seafloor_age(
        global_bounds, data_dir, download=global_download_age_enabled,
    )
    global_relief = load_earth_relief(
        global_bounds, data_dir, download=global_download_relief_enabled,
    )
    global_plate_boundaries = load_plate_boundaries(
        data_dir, download=global_download_boundaries_enabled,
        timeout=network_timeout_s,
    )
    for field, grid_files in global_grids.items():
        global_figure = plot_slab2_global(
            grid_files, field=field, seafloor_age=global_seafloor_age,
            relief=global_relief, plate_boundaries=global_plate_boundaries,
        )
        figure_dir.mkdir(parents=True, exist_ok=True)
        for suffix in ("png", "pdf"):
            figure_file = figure_dir / f"global_slab2_{field}.{suffix}"
            global_figure.savefig(figure_file)
            print(f"Saved: {figure_file}")
        print(f"Plotted {field} for {len(grid_files)} slab regions.")
        global_figure.show()
else:
    print("Enable global_download_enabled to prepare the all-region manifest.")